## Generate evaluation set

In [3]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")
ground_truth[:5]

[{'question': 'I just found this course late — can I still sign up and take part?',
  'document': '74eb249bbf'},
 {'question': 'Am I allowed to join the course after it already started?',
  'document': '74eb249bbf'},
 {'question': 'If I start the course now, is there still a chance to get the certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to be eligible for a certificate if I joined late?',
  'document': '74eb249bbf'},
 {'question': 'Is late enrollment okay, or is the project deadline already closed?',
  'document': '74eb249bbf'}]

In [4]:
# Load docs and build index
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [5]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

doc_idx

{'74eb249bbf': {'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 '977bf7786c': {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 '489dd1c9d9': {'id': '489dd1c9d9',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'What is the video/zoom link to the stream for the “Office Hours

In [6]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [7]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

In [8]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes, you can still join. If you want to receive a certificate, make sure you submit your project while submissions are still being accepted.'

In [9]:
assistant.total_cost()

0.0004987500000000001

In [10]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [ ]:
# Put LLM anwser and original anwser in one record
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'I just found this course late — can I still sign up and take part?',
 'answer_llm': 'Yes, you can still join. If you want to receive a certificate, make sure you submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [ ]:
# Put anwser comparison in a function

def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'I just found this course late — can I still sign up and take part?',
 'answer_llm': 'Yes — you can still join. If you want a certificate, make sure to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [13]:
# Reset token usage before generating all records
assistant.reset_usage()

In [15]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

# Generate anwsers for all ground truth questions
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/565 [00:00<?, ?it/s]

In [16]:
answers = []

for answer_record in results:
    answers.append(answer_record)

In [17]:
assistant.total_cost()

0.6119572500000002

In [18]:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-new.csv", index=False)

In [19]:
df_answers

,question,answer_llm,answer_orig,document
0,I just found this course late — can I still si...,Yes — you can still join. If you want to recei...,"Yes, but if you want to receive a certificate,...",74eb249bbf
1,Am I allowed to join the course after it alrea...,Yes — you can still join after the course has ...,"Yes, but if you want to receive a certificate,...",74eb249bbf
2,"If I start the course now, is there still a ch...","Yes — you can start now, but to get the certif...","Yes, but if you want to receive a certificate,...",74eb249bbf
3,What do I need to do to be eligible for a cert...,I don’t know.,"Yes, but if you want to receive a certificate,...",74eb249bbf
4,"Is late enrollment okay, or is the project dea...",Late enrollment is **not** addressed in the co...,"Yes, but if you want to receive a certificate,...",74eb249bbf
...,...,...,...,...
560,Why am I getting a 401 Client Error when tryin...,Yes — a 401 Client Error usually points to an ...,"If you encounter a 401 Client Error, it may in...",4b30b918bc
561,What should I do if my requests install keeps ...,Install the newer `requests` version directly ...,"If you encounter a 401 Client Error, it may in...",4b30b918bc
562,Is there a way to force pip to install request...,Yes — use the direct source zip URL with a ver...,"If you encounter a 401 Client Error, it may in...",4b30b918bc
563,"Could this error mean my API key is wrong, or ...",It could be either.\n\nA `401 Unauthorized` us...,"If you encounter a 401 Client Error, it may in...",4b30b918bc


## Evaluate RAG

In [1]:
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

# Pydantic model defining LLM as a judge output format
class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [4]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [5]:
# Prompt template (what we send to LLM-as-a-judge)
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [6]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

In [7]:
# Test for 1 record
rec = answers[0]

prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

In [8]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key meaning of the ground truth: late enrollment is allowed, and certificate eligibility depends on submitting the project before submissions close. This is semantically equivalent.', score='good')

In [9]:
calc_price(usage)

{'input_cost': 0.00022125, 'output_cost': 0.0002295, 'total_cost': 0.00045075}

In [11]:
# Wrap the evaluation logic into the function

def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [12]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key message: late enrollment is allowed, and certificate eligibility depends on submitting the project before submissions close. This is semantically equivalent to the ground truth.', score='good')

In [13]:
# Function to run evalutaion on records generated from ground truth
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [14]:
from concurrent.futures import ThreadPoolExecutor

# Run eval for all records in parallel
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/565 [00:00<?, ?it/s]

In [15]:
# Split eval and usage results
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [16]:
df_eval = pd.DataFrame(evaluations)

In [17]:
calc_total_price(usages)

0.4033484999999996

In [18]:
# Aggregate evaluation results
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 547/565 = 96.81%


In [ ]:
# Review bad anwsers
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
3,What do I need to do to be eligible for a cert...,74eb249bbf,bad,The AI answer does not convey the ground truth...
82,What happens if I miss a homework assignment i...,cdc3b285e5,bad,The AI answer does not address the question an...
104,How do I check which OpenAI model I’m actually...,152af39a53,bad,The AI answer is not semantically equivalent t...
161,Is there a simple way to auto-load my `.env` f...,8b2f5e9d04,bad,The AI answer does not match the ground truth....
212,"Should I just catch 429s after they happen, or...",8e21752415,bad,The AI answer does not address the question at...


In [20]:
df_eval.to_csv("data/rag-evaluations-new.csv", index=False)